# 03 · 场景、任务与轨迹
使用模块二的同一个 oven4 资产，生成有界布局，将开关烤箱门指令绑定到具体关节，并通过 MuJoCo 实际执行验证轨迹。

当前语言方法是公开、可检查的开关关节模板。其他语言或未指定的实体/关节会明确报错，不能用模板能力冒充通用语言规划。

In [ ]:
import json
import uuid

from examples._display import setup_notebook

PROJECT_ROOT, OUTPUT_DIR = setup_notebook("scene_and_task_generation")

from physrealx import PhysRealX, SceneSpec, TaskSpec, VersionedRef

px = PhysRealX.open(PROJECT_ROOT / ".physrealx")
run_key = uuid.uuid4().hex[:12]


## 1. 物理资产

In [ ]:
upstream = json.loads(
    (PROJECT_ROOT / "examples/physical_asset_building/output/results.json").read_text()
)
asset_ref = VersionedRef.model_validate(upstream["asset"])
asset = px.assets.get(asset_ref)
print(asset.display_name)


## 2. 布局生成

In [ ]:
layout_operation = px.scenes.generate(
    {
        "resource_id": f"oven4.tabletop.{run_key}",
        "version": "1.0.0",
        "display_name": "Oven4 tabletop",
        "domain": "tabletop",
        "objects": [{"name": "oven", "asset": asset_ref.model_dump(mode="json")}],
        "xy_bounds_m": [-0.4, -0.4, 0.4, 0.4],
        "clearance_m": 0.05,
        "seed": 2026,
    },
    method="bounded_layout",
)
print(dict(layout_operation.metrics))


## 3. 场景装配

In [ ]:
from physrealx import AssetSpec

composition = px.scenes.compose(
    layout_operation.output("scene_draft"), method="collision_constraints"
)
layout = SceneSpec.model_validate_json(px.artifacts.get_bytes(composition.output("scene_spec")))
pusher = AssetSpec.model_validate_json(
    (PROJECT_ROOT / "examples/scene_and_task_generation/input/linear_stage.json").read_bytes()
)
pusher_ref = px.assets.register(pusher)
door = next(j for j in asset.joints if j.name == "e_door_197.joint")
door_initial = 1.0
scene = px.scenes.place_linear_actuator(
    layout,
    entity="oven",
    joint=door.name,
    actuator=pusher_ref,
    name="pusher",
    initial_position=door_initial,
    clearance_m=0.05,
    resource_id=f"oven4.robot-contact.{run_key}",
)
scene_ref = px.scenes.register(scene)
print("场景:", scene_ref.resource_id, "机器人:", pusher_ref.resource_id)


## 4. 任务编译

In [ ]:
task_operation = px.tasks.compile(
    "通过机器人推杆接触关闭烤箱门",
    scene_ref,
    method="articulated_templates",
    config={
        "resource_id": f"oven4.push-close.{run_key}",
        "entity": "oven",
        "joint": door.name,
        "target_fraction": 0.985,
        "tolerance": 0.025,
        "initial_fraction": door_initial / (door.upper_limit - door.lower_limit),
        "max_steps": 160,
        "actuation": {
            "entity": "pusher",
            "joint": "extension",
            "initial_position": 0.0,
            "target_position": scene.metadata["actuator_placement"]["closing_travel_m"],
        },
    },
)
task = TaskSpec.model_validate_json(px.artifacts.get_bytes(task_operation.output("task_spec")))
task_ref = px.tasks.register(task)
print(task.instruction, "受控关节:", task.metadata["controlled_joints"])
print("物体目标判据:", task.success_conditions)


## 5. 约束校核

In [ ]:
validation = px.tasks.validate(task_ref, method="joint_constraints")
print(json.loads(px.artifacts.get_bytes(validation.output("validation_report"))))


## 6. 轨迹生成

In [ ]:
backend_config = {
    "controlled_joints": task.metadata["controlled_joints"],
    "initial_positions": task.metadata["initial_positions"],
    "effort_limits": task.metadata["effort_limits"],
    "passive_positions": task.metadata["passive_positions"],
    "position_gain": 800.0,
    "timestep_seconds": 0.002,
    "substeps": 25,
}
task_instance = px.artifacts.put_json(
    {
        "schema": "physrealx.task-instance/1",
        "task": task_ref.model_dump(mode="json"),
        "backend_config": backend_config,
        "target_positions": task.metadata["target_positions"],
    }
)
trajectory_operation = px.trajectories.generate(
    task_instance,
    method="mujoco_joint_trajectory",
    config={
        "duration_seconds": 6.0,
        "settle_seconds": 1.0,
        "max_tracking_error": 0.05,
        "max_contact_force_n": 100.0,
    },
)
trajectory = trajectory_operation.output("trajectory")
print("真实引擎轨迹验证:", dict(trajectory_operation.metrics))


## 7. 结果导出

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plan = json.loads(px.artifacts.get_bytes(trajectory))
time = (np.arange(len(plan["commands"])) + 1) * plan["dt_seconds"]
figure, ax = plt.subplots(figsize=(7, 3))
ax.plot(time, np.array(plan["commands"])[:, 0], label="Command")
ax.plot(time, np.array(plan["observed_positions"])[:, 0], label="MuJoCo position")
ax.set(xlabel="Time / s", ylabel="Joint position / m")
ax.legend()
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "trajectory.png", dpi=140)
plt.show()

results = {
    "scene": scene_ref.model_dump(mode="json"),
    "task": task_ref.model_dump(mode="json"),
    "trajectory": trajectory.model_dump(mode="json"),
    "backend_config": backend_config,
}
results_ref = px.artifacts.export(OUTPUT_DIR, results)
print("结果索引:", OUTPUT_DIR / "results.json")
